# 文埋め込みとクラスタリングでトピックを作る (BERTopic 方式)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/09_sentence_embeddings/embedding_topics.ipynb)

NLP クックブック 09-04 ／ 文埋め込みとクラスタリング

単語の共起ではなく、文の意味に基づいてトピックを作りたい。クラスタごとに特徴的な語を取り出して名前を付けたい。

In [1]:
!pip install -q fugashi unidic-lite umap-learn

In [2]:
import io, tarfile, urllib.request
import pandas as pd
from sentence_transformers import SentenceTransformer

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor().sample(1500, random_state=0).reset_index(drop=True)
model = SentenceTransformer("intfloat/multilingual-e5-small")
emb = model.encode(["query: " + t for t in df["title"]], normalize_embeddings=True, batch_size=64, show_progress_bar=False)

In [3]:
import numpy as np
import umap
from fugashi import Tagger
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

# 1. 次元削減してからクラスタリング (高次元のままだと距離が効きにくい)
reduced = umap.UMAP(n_neighbors=15, n_components=5, metric="cosine", random_state=0).fit_transform(emb)
df["cluster"] = KMeans(n_clusters=12, n_init=10, random_state=0).fit_predict(reduced)

# 2. クラスタごとに文書を連結して 1 つの大きな文書とみなし、クラスタ単位の TF-IDF (c-TF-IDF) で特徴語を選ぶ
tagger = Tagger()
tok = lambda s: [w.surface for w in tagger(s) if w.feature.pos1 == "名詞" and w.feature.pos2 in ("普通名詞", "固有名詞") and len(w.surface) > 1]
joined = df.groupby("cluster")["title"].apply(" ".join)
vec = CountVectorizer(tokenizer=tok, token_pattern=None, lowercase=False)
counts = vec.fit_transform(joined).toarray()
tf = counts / counts.sum(axis=1, keepdims=True)
idf = np.log(1 + counts.sum(axis=1).mean() / counts.sum(axis=0))
ctfidf = tf * idf
vocab = vec.get_feature_names_out()

for c in joined.index:
    top = [vocab[i] for i in ctfidf[c].argsort()[::-1][:7]]
    main = df[df.cluster == c]["label"].value_counts()
    print(f"{c:2} ({(df.cluster == c).sum():3} 件, 最多カテゴリ {main.index[0]} {main.iloc[0] / main.sum():.0%}) {' '.join(top)}")

 0 (113 件, 最多カテゴリ peachy 41%) ランキング 週間 ゴルフ 恋愛 特集 cafe by
 1 (167 件, 最多カテゴリ sports-watch 87%) Watch Sports 監督 選手 ノム 日本 代表
 2 (101 件, 最多カテゴリ smax 53%) アプリ iPhone Android チャンス ショップ Google ねこ
 3 (151 件, 最多カテゴリ movie-enter 85%) 映画 映像 予告 公開 解禁 週末 読み
 4 ( 89 件, 最多カテゴリ dokujo-tsushin 83%) 独女 女子 結婚 オトナ コミック 恋愛 悩み
 5 (135 件, 最多カテゴリ smax 86%) ドコモ NTT 開始 ソフトウェア 提供 搭載 更新
 6 (195 件, 最多カテゴリ topic-news 64%) ネット 韓国 AKB 話題 発言 殺到 日本
 7 (143 件, 最多カテゴリ it-life-hack 62%) デジ 話題 虎の巻 活用 IT Google PC
 8 (142 件, 最多カテゴリ peachy 61%) プレゼント 終了 クリスマス 開催 招待 ケーキ 試写
 9 ( 90 件, 最多カテゴリ kaden-channel 79%) 売れ筋 チェック ビデオ SALON 発売 パナソニック カメラ
10 ( 89 件, 最多カテゴリ movie-enter 35%) インタビュー オトナ 結婚 映画 女子 エリカ 人生
11 ( 85 件, 最多カテゴリ livedoor-homme 35%) vol 年収 部屋 説教 辛口 話題 図鑑


- この「埋め込み → UMAP → クラスタリング → c-TF-IDF」の流れをまとめたライブラリが BERTopic (`pip install bertopic`)。密度ベースの HDBSCAN でクラスタ数を自動で決め、どこにも属さない外れ値も扱える。
- 短い文 (タイトル、ツイート、アンケートの回答) に特に向く。長い文書は段落に分けてから埋め込む。
- トピックの名前付けは、特徴語と代表文書を LLM に渡して要約させると読みやすくなる。